In [29]:
import pandas as pd
import numpy as np

In [2]:
tours = pd.read_csv('./data/tour.csv')
tour_linked_trips = pd.read_csv('./data/trip_linked.csv')
obs_trips = pd.read_csv('./data/od_20240612_sandagca_weighted_combined_draftfinal.csv')
hts_tour = pd.read_csv('./data/tmodeProfile_vis.csv')
hts_trip = pd.read_csv('./data/tripModeProfile_vis.csv')

C:\Users\ali.etezady\AppData\Local\Temp\ipykernel_1764\2010796033.py:3: DtypeWarning: Columns (17,32,35,36,37,39,40,57,61,62,64,65,79,81,95,97,115,117,119,121,194) have mixed types. Specify dtype option on import or set low_memory=False.
  obs_trips = pd.read_csv('./data/od_20240612_sandagca_weighted_combined_draftfinal.csv')


# Process tour sample data

In [5]:
tour_purpose_reverse_map = {
    1: 'WORK',
    2: 'WORK_RELATED',
    3: 'COLLEGE',
    4: 'SCHOOL',
    5: 'MEDICAL',
    6: 'SHOP',
    7: 'MEAL',
    8: 'SOCIAL',
    9: 'REC',
    10: 'EVENT',
    11: 'ESCORT',
    12: 'AIRPORT',
    13: 'ERRAND',
    14: 'HOTEL',
    15: 'HOME',
    16: 'CHANGE_MODE',
    17: 'OTHER',
    18: 'LOOP',
    995: 'MISSING'
}

mode_category_dict = {
    1: "DRIVE_ALONE",
    2: "SHARED2",
    3: "SHARED3",
    4: "WALK",
    5: "BIKE",
    6: "WALK_LOC",
    7: "WALK_PRM",
    8: "WALK_MIX",
    9: "PNR_LOC",
    10: "PNR_PRM",
    11: "PNR_MIX",
    12: "KNR_LOC",
    13: "KNR_PRM",
    14: "KNR_MIX",
    15: "TNC_LOC",
    16: "TNC_PRM",
    17: "TNC_MIX",
    18: "TAXI",
    19: "TNC_SINGLE",
    20: "TNC_SHARED",
    21: "SCH_BUS",
    22: "EBIKE",
    23: "ESCOOTER",
    995: "MISSING"
}

# Apply mapping to tours DataFrame
tours['tour_purpose_recoded'] = tours['tour_purpose'].map(tour_purpose_reverse_map)
tours['tour_mode_recoded'] = tours['tour_mode'].map(mode_category_dict)
tours['tour_mode_recoded'] = tours['tour_mode_recoded'].replace({
    'WALK_LOC': 'WALK-TRANSIT',
    'WALK_PRM': 'WALK-TRANSIT',
    'WALK_MIX': 'WALK-TRANSIT',
    'PNR_LOC': 'PNR-TRANSIT',
    'PNR_PRM': 'PNR-TRANSIT',
    'PNR_MIX': 'PNR-TRANSIT',
    'KNR_LOC': 'KNR-TRANSIT',
    'KNR_PRM': 'KNR-TRANSIT',
    'KNR_MIX': 'KNR-TRANSIT',
    'TNC_LOC': 'TNC-TRANSIT',
    'TNC_PRM': 'TNC-TRANSIT',
    'TNC_MIX': 'TNC-TRANSIT',
})

display(tours['tour_purpose_recoded'].value_counts(normalize=True))
display(tours['tour_mode_recoded'].value_counts(normalize=False))

WORK            0.329714
COLLEGE         0.131293
ERRAND          0.105627
LOOP            0.090819
SHOP            0.065153
WORK_RELATED    0.062192
SOCIAL          0.052320
REC             0.040474
MEAL            0.039487
MEDICAL         0.030602
SCHOOL          0.025666
OTHER           0.014808
EVENT           0.004936
ESCORT          0.002962
HOTEL           0.001974
AIRPORT         0.001974
Name: tour_purpose_recoded, dtype: float64

WALK-TRANSIT    555
WALK            208
SHARED2          72
KNR-TRANSIT      44
DRIVE_ALONE      39
PNR-TRANSIT      38
TNC_SINGLE       17
TNC-TRANSIT      15
BIKE             14
ESCOOTER          5
EBIKE             4
TAXI              2
Name: tour_mode_recoded, dtype: int64

In [6]:
tour_linked_trips = pd.merge(tour_linked_trips, tours[['tour_id', 'tour_purpose_recoded', 'tour_mode_recoded']], on='tour_id', how='left')
tour_linked_trips['linked_trip_mode_name'] = tour_linked_trips['linked_trip_mode'].map(mode_category_dict)
#recode linked_trip_mode_name so walk_loc, walk_prm, walk_mix all become walk_transit, etc.
tour_linked_trips['linked_trip_mode_name'] = tour_linked_trips['linked_trip_mode_name'].replace({
    'WALK_LOC': 'WALK-TRANSIT',
    'WALK_PRM': 'WALK-TRANSIT',
    'WALK_MIX': 'WALK-TRANSIT',
    'PNR_LOC': 'PNR-TRANSIT',
    'PNR_PRM': 'PNR-TRANSIT',
    'PNR_MIX': 'PNR-TRANSIT',
    'KNR_LOC': 'KNR-TRANSIT',
    'KNR_PRM': 'KNR-TRANSIT',
    'KNR_MIX': 'KNR-TRANSIT',
    'TNC_LOC': 'TNC-TRANSIT',
    'TNC_PRM': 'TNC-TRANSIT',
    'TNC_MIX': 'TNC-TRANSIT',
})

purp_map = {
    'WORK': 'Work',
    'ERRAND': 'Other Maintenance',
    'COLLEGE': 'University',
    'WORK_RELATED': 'Other Maintenance',
    'SHOP': 'Other Discretionary',
    'LOOP': 'Other Discretionary',
    'MEAL': 'Other Discretionary',
    'SOCIAL': 'Other Discretionary',
    'MEDICAL': 'Other Discretionary',
    'REC': 'Other Discretionary',
    'SCHOOL': 'School',
    'EVENT': 'Other Discretionary',
    'ESCORT': 'Escort'
}

tour_linked_trips['tourpurp'] = tour_linked_trips['tour_purpose_recoded'].map(purp_map)

tour_linked_trips.tourpurp.value_counts()
# pd.crosstab(tour_linked_trips['tour_purpose_recoded'], tour_linked_trips['tourpurp'], margins=True)


Work                   706
Other Discretionary    550
Other Maintenance      541
University             267
School                  52
Escort                   4
Name: tourpurp, dtype: int64

## check trip vs tour mode distribution

In [7]:
# create a dictionary with crosstab by tourpurp
trip_mode_target = {}
for purp in tour_linked_trips.tourpurp.unique():
    if pd.isna(purp) or purp is None:
        continue
    print(purp)
    df_purp = tour_linked_trips[tour_linked_trips.tourpurp == purp]
    if df_purp.empty:
        continue
    xtab = pd.crosstab(df_purp['linked_trip_mode_name'], df_purp['tour_mode_recoded'], margins=True)
    trip_mode_target[purp] = xtab

Work
University
Other Discretionary
Other Maintenance
School
Escort


## get trips/tour rates

In [8]:
tours_endingathome = tour_linked_trips.groupby('tour_id').filter(lambda x: x.iloc[-1]['d_purpose_category'] == 15)['tour_id'].unique()
tours_withmorethan1trip = tour_linked_trips.groupby('tour_id').filter(lambda x: len(x) > 1)['tour_id'].unique()
tours_to_keep = np.intersect1d(tours_endingathome, tours_withmorethan1trip)
print(f'number of closed-jaw tours: {len(tours_to_keep)}')
print(f'total number of tours: {tour_linked_trips["tour_id"].nunique()}')
tour_linked_trips_to_keep = tour_linked_trips[tour_linked_trips['tour_id'].isin(tours_to_keep)]

number of closed-jaw tours: 377
total number of tours: 1013


In [9]:
def calculate_trip_rates_per_tour(abms_trips_df, tour_mode_list):
    result_df = pd.DataFrame(columns=['transit_trips_per_tour'])
    for tour_mode in tour_mode_list:
        tours = abms_trips_df[abms_trips_df['tour_mode_recoded'] == tour_mode]['tour_id'].nunique()
        all_trips = abms_trips_df[abms_trips_df['tour_mode_recoded'] == tour_mode].shape[0]
        transit_trips = abms_trips_df[abms_trips_df['linked_trip_mode_name'] == tour_mode].shape[0]
        transit_trips_per_tour = transit_trips / tours if tours > 0 else float('nan')
        print(f"Calculating for tour mode: {tour_mode}")
        print(f"Tours: {tours}")
        print(f"All trips: {all_trips}")
        print(f"Transit trips: {transit_trips}")
        print(f"Transit Trips/Tour: {round(transit_trips_per_tour,2)}")
        print('-----------')
        result_df.loc[tour_mode] = round(transit_trips_per_tour, 2)
    result_df.index.name = 'tour_mode'
    return result_df

transit_trips_per_transit_tour_df = calculate_trip_rates_per_tour(tour_linked_trips_to_keep, ['WALK-TRANSIT', 'PNR-TRANSIT', 'KNR-TRANSIT', 'TNC-TRANSIT'])
# set trips/tour to 2 for PNR, KNR, TNC since acitivutysim cannot accept stops on these tours
for mode in ['PNR-TRANSIT', 'KNR-TRANSIT', 'TNC-TRANSIT']:
    transit_trips_per_transit_tour_df.loc[mode, 'transit_trips_per_tour'] = 2

Calculating for tour mode: WALK-TRANSIT
Tours: 217
All trips: 636
Transit trips: 484
Transit Trips/Tour: 2.23
-----------
Calculating for tour mode: PNR-TRANSIT
Tours: 20
All trips: 67
Transit trips: 31
Transit Trips/Tour: 1.55
-----------
Calculating for tour mode: KNR-TRANSIT
Tours: 23
All trips: 78
Transit trips: 27
Transit Trips/Tour: 1.17
-----------
Calculating for tour mode: TNC-TRANSIT
Tours: 5
All trips: 10
Transit trips: 7
Transit Trips/Tour: 1.4
-----------


# OBS trips data

## process OBS trip file

In [10]:
# Define mapping dictionary for access modes
obs_trip_mode_map = {
    'Walk': 'Walk',
    'Bike (personal)': 'Walk',
    'E-Bike (personal)': 'Walk',
    'E-Bike (shared)': 'Walk',
    'Skateboard': 'Walk',
    'Wheelchair': 'Walk',
    'E-scooter (personal)': 'Walk',
    'E-scooter (shared)': 'Walk',
    'Drove alone and parked': 'PNR',
    'Drove or rode with others and parked': 'PNR',
    'Was dropped off by someone': 'KNR',
    'Taxi': 'TNC',
    'Other shuttle': 'KNR',
    'Electric vehicle shuttle': 'KNR',
    'Uber, Lyft, etc. (private)': 'TNC',
    'Uber, Lyft, etc. (pool or shared)': 'TNC',
    'Other': None  # or 'Other' if you want to keep as a category
}


# Create new column 'access_mode' in obs_trips DataFrame
obs_trips['access_mode'] = obs_trips['ORIGIN_TRANSPORT'].map(obs_trip_mode_map)
obs_trips['egress_mode'] = obs_trips['DESTIN_TRANSPORT'].map(obs_trip_mode_map)

def determine_mode(row):
    if (row['access_mode'] == 'PNR') | (row['egress_mode'] == 'PNR'):
        return 'PNR-TRANSIT'
    if (row['access_mode'] == 'KNR') | (row['egress_mode'] == 'KNR'):
        return 'KNR-TRANSIT'
    if (row['access_mode'] == 'TNC') | (row['egress_mode'] == 'TNC'):
        return 'TNC-TRANSIT'
    return 'WALK-TRANSIT'

obs_trips['mode'] = obs_trips.apply(lambda row: determine_mode(row), axis=1)
obs_trips['mode'].value_counts()
obs_trips['tourweight'] = obs_trips['LINKED_WGHT_FCTR']

# Adjust weights by mode
obs_trips.loc[obs_trips['mode'] == 'WALK-TRANSIT','tourweight'] = obs_trips.loc[obs_trips['mode'] == 'WALK-TRANSIT', 'tourweight'] / transit_trips_per_transit_tour_df.loc['WALK-TRANSIT', 'transit_trips_per_tour']
obs_trips.loc[obs_trips['mode'] == 'PNR-TRANSIT', 'tourweight'] = obs_trips.loc[obs_trips['mode'] == 'PNR-TRANSIT', 'tourweight'] / 2 #transit_obs_trips_per_transit_tour_df.loc['PNR-TRANSIT', 'transit_obs_trips_per_tour']
obs_trips.loc[obs_trips['mode'] == 'KNR-TRANSIT', 'tourweight'] = obs_trips.loc[obs_trips['mode'] == 'KNR-TRANSIT', 'tourweight'] / 2 #transit_obs_trips_per_transit_tour_df.loc['KNR-TRANSIT', 'transit_obs_trips_per_tour']
obs_trips.loc[obs_trips['mode'] == 'TNC-TRANSIT', 'tourweight'] = obs_trips.loc[obs_trips['mode'] == 'TNC-TRANSIT', 'tourweight'] / 2 #transit_obs_trips_per_transit_tour_df.loc['TNC-TRANSIT', 'transit_obs_trips_per_tour']

print("Total number of On-board survey Tours: ", int(obs_trips['tourweight'].sum()))
print("Total number of On-board survey obs_trips: ", int(obs_trips['LINKED_WGHT_FCTR'].sum()))

Total number of On-board survey Tours:  102467
Total number of On-board survey obs_trips:  225411


### add auto sufficiency status

In [11]:
obs_trips['auto_suff'] = np.where(obs_trips['COUNT_VH_HH[Code]'] == 0, 0, 
                                    np.where(obs_trips['COUNT_VH_HH[Code]'] < obs_trips['HH_SIZE_18UP[Code]'], 1, 2))

### add purpose (code gotten from previous task on obs trip analysis by RSG)

In [12]:
tour_purpose_mapping = {
    1: 'Work',
    2: 'College/University',
    3: 'K-12',
    4: 'Escort',
    5: 'Ind_Shop',
    6: 'Ind_Other_Maint',
    7: 'Ind_Eat_Out',
    7.5: 'Ind_Soc_Visit',
    8: 'Ind_Other_Disc',
    9: 'Work_Based',
    10: 'Res-Airport',
    10.5: 'Ext-Airport',
    11: 'NonRes-Airport',
    12: 'Visitor_Work',
    13: 'Visitor_Rec',
    14: 'Visitor_EatOut',
    15: 'MR_Work',
    16: 'MR_School',
    17: 'MR_Shop',
    18: 'MR_Visit',
    19: 'MR_Other',
    21: 'ExtInt_Work',
    22: 'ExtInt_NonWork',
    23: 'Special_Event',
    24: 'Jnt_Shop',
    25: 'Jnt_Other_Maint',
    26: 'Jnt_Eat_Out',
    27: 'Jnt_Other_Disc',
    28: 'Jnt_Soc_Visit'
}

final_purpose_map = {
    'Work': 'Work',
    'College/University': 'University',
    'K-12': 'School',
    # 'Escort': 'Joint-Discretionary',
    'Ind_Shop': 'Ind-Discretionary',
    'Ind_Other_Maint': 'Ind-Maintenance',
    'Ind_Eat_Out': 'Ind-Discretionary',
    'Ind_Soc_Visit': 'Ind-Discretionary',
    'Ind_Other_Disc': 'Ind-Discretionary',
    'Work_Based': 'AtWork',
    'Res-Airport': 'Res-Airport',
    'Ext-Airport': 'Ext-Airport',
    'NonRes-Airport': 'NonRes-Airport',
    'Visitor_Work': 'Visitor-Work',
    'Visitor_Rec': 'Visitor-Discretionary',
    'Visitor_EatOut': 'Visitor-EatOut',
    'MR_Work': 'MR_Work',
    'MR_School': 'MR_School',
    'MR_Shop': 'MR_Discretionary',
    'MR_Visit': 'MR_Discretionary',
    'MR_Other': 'MR_Discretionary',
    'ExtInt_Work': 'ExtInt_Work',
    'ExtInt_NonWork': 'ExtInt_NonWork Discretionary',
    'Special_Event': 'Ind-Discretionary',
    'Jnt_Shop': 'Joint-Discretionary',
    'Jnt_Other_Maint': 'Joint-Maintenance',
    'Jnt_Eat_Out': 'Joint-Discretionary',
    'Jnt_Other_Disc': 'Joint-Discretionary',
    'Jnt_Soc_Visit': 'Joint-Discretionary'
}                               
# Initialize tourPurpose to 0
obs_trips['tourPurpose[Code]'] = 0

# Apply the conditions for tourPurpose assignment
# 1 = Work
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 1) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 1
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 1) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 1
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 1) & (obs_trips['RESIDENT_VISITOR[Code]'].isin([4, 98])), 'tourPurpose[Code]'] = 1
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 1) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['RESIDENT_VISITOR[Code]'].isin([4, 98])), 'tourPurpose[Code]'] = 1
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] != 1) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] != 1) & (obs_trips['DID_U_GO2WORK[Code]'] == 1) & (obs_trips['WILL_GO2WORK[Code]'] == 2) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 1
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] != 1) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] != 1) & (obs_trips['DID_U_GO2WORK[Code]'] == 2) & (obs_trips['WILL_GO2WORK[Code]'] == 1) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 1
# 2 = College\University
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 5) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 2
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 5) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 2
## exchange students
obs_trips.loc[(obs_trips['DESTIN_PLACE_TYPE[Code]'] == 5) & (obs_trips['RESIDENT_VISITOR[Code]'] != 1) & ((obs_trips['USA_OR_MEXICO[Code]'] == 1)), 'tourPurpose[Code]'] = 2
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 5) & (obs_trips['RESIDENT_VISITOR[Code]'] != 1) & ((obs_trips['USA_OR_MEXICO[Code]'] == 1)), 'tourPurpose[Code]'] = 2
# 3 = K-12
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 6) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 3
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 6) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 3
# 4 = Escort
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 14) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 4
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 14) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 4
# 5 = Ind_Shop
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 8) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 5
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 8) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 5
# 6 = Ind_Other_Maint
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 4) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 6
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 4) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 6
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 7) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 6
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 7) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 6
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 15) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 6
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 15) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 6
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 98) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 6
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 98) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 6
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 99) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 6
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 99) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 6
# 7 = Ind_Eat_Out
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 9) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 7
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 9) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 7
# 7.5 = Ind_Soc_Visit
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 10) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 7.5
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 10) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 7.5
# 8 = Ind_Other_Disc
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 11) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 8
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 11) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 8
# homeless
obs_trips.loc[(obs_trips['DESTIN_PLACE_TYPE[Code]'] == 22) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 8 
obs_trips.loc[(obs_trips['DESTIN_PLACE_TYPE[Code]'] == 1) & (obs_trips['RESIDENT_VISITOR[Code]'] == 22), 'tourPurpose[Code]'] = 8 
# 9 - Work_Based
obs_trips.loc[(obs_trips['RESIDENT_VISITOR[Code]'] == 1) & (obs_trips['DID_U_GO2WORK[Code]'] == 1) & (obs_trips['WILL_GO2WORK[Code]'] == 1), 'tourPurpose[Code]'] = 9

# 10 = Res-Airport
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 12) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 10
obs_trips.loc[(obs_trips['DESTIN_PLACE_TYPE[Code]'] == 12) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 10
# 10.5 = Ext-Airport
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 12) & (obs_trips['RESIDENT_VISITOR[Code]'] == 2), 'tourPurpose[Code]'] = 10.5
obs_trips.loc[(obs_trips['DESTIN_PLACE_TYPE[Code]'] == 12) & (obs_trips['RESIDENT_VISITOR[Code]'] == 2), 'tourPurpose[Code]'] = 10.5
# 11 = NonRes-Airport
obs_trips.loc[(obs_trips['DESTIN_PLACE_TYPE[Code]'] == 12) & (~obs_trips['RESIDENT_VISITOR[Code]'].isin([1,2])), 'tourPurpose[Code]'] = 11
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 12) & (~obs_trips['RESIDENT_VISITOR[Code]'].isin([1,2])), 'tourPurpose[Code]'] = 11
# 12 = Visitor_Work
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 3) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 1) & (~obs_trips['RESIDENT_VISITOR[Code]'].isin([1, 2, 3])), 'tourPurpose[Code]'] = 12
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 3) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 5) & (~obs_trips['RESIDENT_VISITOR[Code]'].isin([1, 2, 3])), 'tourPurpose[Code]'] = 12
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 3) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 6) & (~obs_trips['RESIDENT_VISITOR[Code]'].isin([1, 2, 3])), 'tourPurpose[Code]'] = 12
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 1) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 3) & (~obs_trips['RESIDENT_VISITOR[Code]'].isin([1, 2, 3])), 'tourPurpose[Code]'] = 12
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 5) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 3) & (~obs_trips['RESIDENT_VISITOR[Code]'].isin([1, 2, 3])), 'tourPurpose[Code]'] = 12
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 6) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 3) & (~obs_trips['RESIDENT_VISITOR[Code]'].isin([1, 2, 3])), 'tourPurpose[Code]'] = 12
# 13 = Visitor_Rec
obs_trips.loc[(~obs_trips['ORIGIN_PLACE_TYPE[Code]'].isin([1,5,6,9,12,13])) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 3) & (~obs_trips['RESIDENT_VISITOR[Code]'].isin([1, 2, 3])), 'tourPurpose[Code]'] = 13
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 3) & (~obs_trips['DESTIN_PLACE_TYPE[Code]'].isin([1,5,6,9,12,13])) & (~obs_trips['RESIDENT_VISITOR[Code]'].isin([1, 2, 3])), 'tourPurpose[Code]'] = 13
obs_trips.loc[(~obs_trips['DESTIN_PLACE_TYPE[Code]'].isin([1,2,3,5,6,9,12,13])) & (~obs_trips['DESTIN_PLACE_TYPE[Code]'].isin([1,2,3,5,6,9,12,13])) & (~obs_trips['RESIDENT_VISITOR[Code]'].isin([1, 2, 3])), 'tourPurpose[Code]'] = 13
# 14 = Visitor_EatOut
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 3) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 9) & (~obs_trips['RESIDENT_VISITOR[Code]'].isin([1, 2, 3])), 'tourPurpose[Code]'] = 14
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 9) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 3) & (~obs_trips['RESIDENT_VISITOR[Code]'].isin([1, 2, 3])), 'tourPurpose[Code]'] = 14
# 15 = MR_Work
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 1) & ((obs_trips['RESIDENT_VISITOR[Code]'] == 3) | (obs_trips['USA_OR_MEXICO[Code]'] == 2)), 'tourPurpose[Code]'] = 15
obs_trips.loc[(obs_trips['DESTIN_PLACE_TYPE[Code]'] == 1) & ((obs_trips['RESIDENT_VISITOR[Code]'] == 3) | (obs_trips['USA_OR_MEXICO[Code]'] == 2)), 'tourPurpose[Code]'] = 15
# 16 = MR_School
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'].isin([5,6])) & ((obs_trips['RESIDENT_VISITOR[Code]'] == 3) | (obs_trips['USA_OR_MEXICO[Code]'] == 2)), 'tourPurpose[Code]'] = 16
obs_trips.loc[(obs_trips['DESTIN_PLACE_TYPE[Code]'].isin([5,6])) & ((obs_trips['RESIDENT_VISITOR[Code]'] == 3) | (obs_trips['USA_OR_MEXICO[Code]'] == 2)), 'tourPurpose[Code]'] = 16
# 17 = MR_Shop
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 8) & ((obs_trips['RESIDENT_VISITOR[Code]'] == 3) | (obs_trips['USA_OR_MEXICO[Code]'] == 2)), 'tourPurpose[Code]'] = 17
obs_trips.loc[(obs_trips['DESTIN_PLACE_TYPE[Code]'] == 8) & ((obs_trips['RESIDENT_VISITOR[Code]'] == 3) | (obs_trips['USA_OR_MEXICO[Code]'] == 2)), 'tourPurpose[Code]'] = 17
# 18 = MR_Visit
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 10) & ((obs_trips['RESIDENT_VISITOR[Code]'] == 3) | (obs_trips['USA_OR_MEXICO[Code]'] == 2)), 'tourPurpose[Code]'] = 18
obs_trips.loc[(obs_trips['DESTIN_PLACE_TYPE[Code]'] == 10) & ((obs_trips['RESIDENT_VISITOR[Code]'] == 3) | (obs_trips['USA_OR_MEXICO[Code]'] == 2)), 'tourPurpose[Code]'] = 18
# 19 = MR_Other **
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (~obs_trips['ORIGIN_PLACE_TYPE[Code]'].isin([1, 5, 6, 8, 10])) & ((obs_trips['RESIDENT_VISITOR[Code]'] == 3) | (obs_trips['USA_OR_MEXICO[Code]'] == 2)), 'tourPurpose[Code]'] = 19
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (~obs_trips['DESTIN_PLACE_TYPE[Code]'].isin([1, 5, 6, 8, 10])) & ((obs_trips['RESIDENT_VISITOR[Code]'] == 3) | (obs_trips['USA_OR_MEXICO[Code]'] == 2)), 'tourPurpose[Code]'] = 19
# 21 = ExtInt_Work
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 1) & (obs_trips['RESIDENT_VISITOR[Code]'] == 2), 'tourPurpose[Code]'] = 21
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 1) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['RESIDENT_VISITOR[Code]'] == 2), 'tourPurpose[Code]'] = 21
# 22 = ExtInt_NonWork
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 2) & (~obs_trips['DESTIN_PLACE_TYPE[Code]'].isin([1, 12, 13])) & (obs_trips['RESIDENT_VISITOR[Code]'] == 2), 'tourPurpose[Code]'] = 22
obs_trips.loc[(~obs_trips['ORIGIN_PLACE_TYPE[Code]'].isin([1, 12, 13])) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['RESIDENT_VISITOR[Code]'] == 2), 'tourPurpose[Code]'] = 22
# 23 = Special_Event
obs_trips.loc[(obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 13), 'tourPurpose[Code]'] = 23
obs_trips.loc[(obs_trips['DESTIN_PLACE_TYPE[Code]'] == 13), 'tourPurpose[Code]'] = 23

## not coded
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 1) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 1
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 1) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 1

obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 5) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 2
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 5) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 2
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 5), 'tourPurpose[Code]'] = 2
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 5), 'tourPurpose[Code]'] = 2

obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 6) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 3
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 6) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 3
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 6), 'tourPurpose[Code]'] = 3
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 6), 'tourPurpose[Code]'] = 3

obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 14) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 4
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 14) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 4

obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 8) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 5
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 8) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 5

obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 4) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 6
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 4) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 6
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 7) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 6
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 7) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 6
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 15) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 6
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 15) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 6

obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 9) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 7
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 9) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 7

obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 10) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 8
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 10) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 8
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 11) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 8
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 11) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 8
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 3) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 8
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 3) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 2) & (obs_trips['RESIDENT_VISITOR[Code]'] == 1), 'tourPurpose[Code]'] = 8

obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 1)  & (~obs_trips['RESIDENT_VISITOR[Code]'].isin([1, 2, 3])), 'tourPurpose[Code]'] = 12
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 1)  & (~obs_trips['RESIDENT_VISITOR[Code]'].isin([1, 2, 3])), 'tourPurpose[Code]'] = 12

obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (~obs_trips['ORIGIN_PLACE_TYPE[Code]'].isin([1,5,6,9,12,13])) & (~obs_trips['DESTIN_PLACE_TYPE[Code]'].isin([1,5,6,9,12,13])) & (~obs_trips['RESIDENT_VISITOR[Code]'].isin([1, 2, 3])), 'tourPurpose[Code]'] = 13

obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 9)  & (~obs_trips['RESIDENT_VISITOR[Code]'].isin([1, 2, 3])), 'tourPurpose[Code]'] = 14
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 9)  & (~obs_trips['RESIDENT_VISITOR[Code]'].isin([1, 2, 3])), 'tourPurpose[Code]'] = 14

obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['DESTIN_PLACE_TYPE[Code]'] == 1) & (obs_trips['RESIDENT_VISITOR[Code]'] == 2), 'tourPurpose[Code]'] = 21
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (obs_trips['ORIGIN_PLACE_TYPE[Code]'] == 1) & (obs_trips['RESIDENT_VISITOR[Code]'] == 2), 'tourPurpose[Code]'] = 21

obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 0) & (~obs_trips['ORIGIN_PLACE_TYPE[Code]'].isin([1, 12, 13])) & (~obs_trips['DESTIN_PLACE_TYPE[Code]'].isin([1, 12, 13])) & (obs_trips['RESIDENT_VISITOR[Code]'] == 2), 'tourPurpose[Code]'] = 22

obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 5) & (obs_trips['HOW_MANY_PPL_TRAVEL[Code]'] > 0) & (obs_trips['TRAVEL_HH[Code]'] > 0), 'tourPurpose[Code]'] = 24
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 6) & (obs_trips['HOW_MANY_PPL_TRAVEL[Code]'] > 0) & (obs_trips['TRAVEL_HH[Code]'] > 0), 'tourPurpose[Code]'] = 25
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 7) & (obs_trips['HOW_MANY_PPL_TRAVEL[Code]'] > 0) & (obs_trips['TRAVEL_HH[Code]'] > 0), 'tourPurpose[Code]'] = 26
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 8) & (obs_trips['HOW_MANY_PPL_TRAVEL[Code]'] > 0) & (obs_trips['TRAVEL_HH[Code]'] > 0), 'tourPurpose[Code]'] = 27

# 28 = Jnt_Soc_Visit
obs_trips.loc[(obs_trips['tourPurpose[Code]'] == 7.5) & (obs_trips['HOW_MANY_PPL_TRAVEL[Code]'] > 0) & (obs_trips['TRAVEL_HH[Code]'] > 0), 'tourPurpose[Code]'] = 28

obs_trips['tourPurpose'] = obs_trips['tourPurpose[Code]'].map(tour_purpose_mapping)
obs_trips['tourPurpose'] = obs_trips['tourPurpose'].map(final_purpose_map).fillna('Other')


## separate target by model type

In [13]:
resident_df = obs_trips[(obs_trips['RESIDENT_VISITOR[Code]'] == 1)&(obs_trips['tourPurpose'] !='Res-Airport')]
visitor_df = obs_trips[obs_trips.tourPurpose.str.startswith('Visitor')]
airport_df = obs_trips[obs_trips.tourPurpose.str.endswith('Airport')]
xborder_df = obs_trips[obs_trips.tourPurpose.str.startswith('MR_')]

## create resident tour target

In [14]:
def create_obs_trip_targets_for_targets_tables(obs_df, purposes, weight):
    dfs = {}
    all_purposes_ct = pd.crosstab(
        obs_df['mode'],
        obs_df['auto_suff'],
        values=obs_df[weight],
        aggfunc='sum',
        margins=True,
        dropna=False
    )
    all_purposes_ct = round(all_purposes_ct.fillna(0),0)
    
    all_purposes_ct.name = 'Total'
    dfs[all_purposes_ct.name] = all_purposes_ct
    
    # Create crosstab for each purpose
    for tour_purpose in purposes:
        print(tour_purpose)
        tour_mode_autosuff_by_purp_ct = pd.crosstab(
            obs_df[(obs_df['tourPurpose'] == tour_purpose)]['mode'],
            obs_df[(obs_df['tourPurpose'] == tour_purpose)]['auto_suff'],
            values=obs_df[weight],
            aggfunc='sum',
            margins=True,
            dropna=False
        )
        # tour_mode_autosuff_by_purp_ct = tour_mode_autosuff_by_purp_ct.reindex(
        #     index=output_calibration_modes, fill_value=0)
        tour_mode_autosuff_by_purp_ct = round(tour_mode_autosuff_by_purp_ct.fillna(0),0)

        tour_mode_autosuff_by_purp_ct.name = tour_purpose
        dfs[tour_mode_autosuff_by_purp_ct.name] = tour_mode_autosuff_by_purp_ct

    return dfs

def create_tour_targets(targets_dict, transit_trips_per_transit_tour_df):
    """
    Adjusts each trip target DataFrame in targets_dict by dividing the number of trips in each row
    by the corresponding factor in transit_trips_per_transit_tour_df, if the row index matches a transit mode.
    Returns a new dictionary with adjusted DataFrames.
    """
    adjusted_targets = {}
    for purpose, df in targets_dict.items():
        df_adj = df.copy()
        for mode in transit_trips_per_transit_tour_df.index:
            if mode in df_adj.index:
                factor = transit_trips_per_transit_tour_df.loc[mode, 'transit_trips_per_tour']
                if factor and factor > 0:
                    df_adj.loc[mode] = df_adj.loc[mode] / factor
                    
        df_adj.loc['All'] = df_adj.loc[[i for i in df_adj.index if i != 'All']].sum()
        adjusted_targets[purpose] = df_adj
        
    return adjusted_targets

resident_allpurp_trip_targets = create_obs_trip_targets_for_targets_tables(resident_df, ['Work', 'University', 'School', 'Ind-Maintenance', 'Ind-Discretionary', 'Joint-Maintenance', 'Ind-Discretionary'], 'LINKED_WGHT_FCTR')
tour_targets = create_tour_targets(resident_allpurp_trip_targets, transit_trips_per_transit_tour_df)

Work
University
School
Ind-Maintenance
Ind-Discretionary
Joint-Maintenance
Ind-Discretionary


In [15]:
resident_allpurp_trip_targets['Total']

auto_suff,0,1,2,All
mode,,,,
KNR-TRANSIT,2555.0,5055.0,2338.0,9947.0
PNR-TRANSIT,189.0,2461.0,3200.0,5850.0
TNC-TRANSIT,1009.0,1443.0,647.0,3098.0
WALK-TRANSIT,69074.0,73652.0,31656.0,174381.0
All,72827.0,82609.0,37840.0,193277.0


In [16]:
#for each purpose, recalculate All
for purpose, df in tour_targets.items():
    if 'All' in df.index:
        df.loc['All'] = df.loc[[i for i in df.index if i != 'All']].sum()

In [17]:
rows = []
for purpose, df in tour_targets.items():
    for tour_mode in df.index:
        row = {
            'tour_mode': tour_mode,
            'tour_purpose': purpose,
            'autosuff==0': df.loc[tour_mode, 0] if 0 in df.columns else None,
            'auto_suff==1': df.loc[tour_mode, 1] if 1 in df.columns else None,
            'auto_suff==2': df.loc[tour_mode, 2] if 2 in df.columns else None,
            'All': df.loc[tour_mode, 'All'] if 'All' in df.columns else None
        }
        rows.append(row)

tour_targets_long = pd.DataFrame(rows)
tour_targets_long.to_csv('./output/resident_tour_target.csv', index=False)
print('Wrote tour_targets_long.csv with columns: tour_mode, tour_purpose, autosuff==1, auto_suff==1, auto_suff==2, All')
display(tour_targets_long.head())

Wrote tour_targets_long.csv with columns: tour_mode, tour_purpose, autosuff==1, auto_suff==1, auto_suff==2, All


,tour_mode,tour_purpose,autosuff==0,auto_suff==1,auto_suff==2,All
0,KNR-TRANSIT,Total,1277.500000,2527.500000,1169.000000,4973.500000
1,PNR-TRANSIT,Total,94.500000,1230.500000,1600.000000,2925.000000
2,TNC-TRANSIT,Total,504.500000,721.500000,323.500000,1549.000000
3,WALK-TRANSIT,Total,30974.887892,33027.802691,14195.515695,78197.757848
4,All,Total,32851.387892,37507.302691,17288.015695,87645.257848


## create resident trip targets

In [18]:
# create a dictionary with crosstab by tourpurp
trip_mode_target = {}
for purp in tour_linked_trips_to_keep.tourpurp.unique():
    if pd.isna(purp) or purp is None:
        continue
    print(purp)
    df_purp = tour_linked_trips_to_keep[tour_linked_trips_to_keep.tourpurp == purp]
    if df_purp.empty:
        continue
    xtab = pd.crosstab(df_purp['linked_trip_mode_name'], df_purp['tour_mode_recoded'], margins=True)
    trip_mode_target[purp] = xtab

Work
University
Other Maintenance
Other Discretionary
School


In [19]:
def no_other_trips_on_pnr_knr_tours(trip_mode_target):
    """Ensure only transit trips on PNR/KNR tours"""
    for purp, df in trip_mode_target.items():
        print(f'Working on puporse: {purp}')
        for tour_mode in ['PNR-TRANSIT', 'KNR-TRANSIT', 'TNC-TRANSIT']:
            if tour_mode in df.columns:
                for trip_mode in df.index:
                    if trip_mode not in [tour_mode, 'All']:
                        other_trips = df.loc[trip_mode, tour_mode]
                        if trip_mode in df.columns:
                            df.loc[trip_mode, tour_mode] = 0
                            df.loc[trip_mode, trip_mode] += other_trips
                            df.loc['All', trip_mode] = df.loc[df.index != 'All', trip_mode].sum()

                        else:
                            df.loc[:, trip_mode] = 0
                            df.loc[trip_mode, trip_mode] += other_trips
                            df.loc['All', trip_mode] = df.loc[df.index != 'All', trip_mode].sum()
                #sum all rows except All

                df.loc['All', tour_mode] = df.loc[df.index != 'All', tour_mode].sum()

    return trip_mode_target

trip_mode_target = no_other_trips_on_pnr_knr_tours(trip_mode_target)

Working on puporse: Work
Working on puporse: University
Working on puporse: Other Maintenance
Working on puporse: Other Discretionary
Working on puporse: School


In [20]:
# Get the weighted number of transit trips by mode and purpose for resident_df
transit_modes = ['WALK-TRANSIT', 'PNR-TRANSIT', 'KNR-TRANSIT', 'TNC-TRANSIT']

weighted_trip_counts = (
    resident_df[resident_df['mode'].isin(transit_modes)]
    .groupby(['mode', 'tourPurpose'])['LINKED_WGHT_FCTR']
    .sum()
    .unstack(fill_value=0)
    .reindex(index=transit_modes, fill_value=0)
)

weighted_trip_counts['All'] = weighted_trip_counts.sum(axis=1)
weighted_trip_counts.loc['All'] = weighted_trip_counts.sum(axis=0)

weighted_trip_counts['Other Maintenance'] = weighted_trip_counts['Ind-Maintenance'] + weighted_trip_counts['Joint-Maintenance']
weighted_trip_counts['Other Discretionary'] = weighted_trip_counts['Ind-Discretionary'] + weighted_trip_counts['Joint-Discretionary']


display(weighted_trip_counts)

tourPurpose,AtWork,Ind-Discretionary,Ind-Maintenance,Joint-Discretionary,Joint-Maintenance,Other,School,University,Work,All,Other Maintenance,Other Discretionary
mode,,,,,,,,,,,,
WALK-TRANSIT,172.313194,40864.950463,23507.647001,9461.141810,3356.963614,1012.427000,8313.796264,25747.072619,61944.981903,174381.293868,26864.610615,50326.092273
PNR-TRANSIT,3.325831,997.643287,557.234674,249.035603,65.522908,15.720822,23.782186,1248.434177,2689.645704,5850.345192,622.757582,1246.678891
KNR-TRANSIT,27.005035,1956.717619,1020.701954,616.123875,285.504621,22.452081,510.632788,1066.475589,4441.507195,9947.120757,1306.206575,2572.841494
TNC-TRANSIT,21.619327,924.470656,435.811741,362.266225,94.824990,0.000000,33.190920,164.885477,1061.114960,3098.184297,530.636732,1286.736881
All,224.263387,44743.782025,25521.395371,10688.567513,3802.816133,1050.599902,8881.402159,28226.867861,70137.249762,193276.944113,29324.211504,55432.349538


In [21]:
# Transit Mode Tour Scaling Function
def scale_trip_targets_to_match_weighted_counts(trip_mode_target, weighted_trip_counts):

    scaled_targets = {}
    transit_modes = ['WALK-TRANSIT', 'PNR-TRANSIT', 'KNR-TRANSIT', 'TNC-TRANSIT']
    for purpose, target_df in trip_mode_target.items():
            print(f"------------------------------------------------")
            print(f"Scaling targets for purpose: {purpose}")
            # Create a copy of the target DataFrame
            df_scaled = target_df.copy()
            
            # Scale each transit mode
            for transit_mode in transit_modes:
                if transit_mode in df_scaled.columns:
                    print(f'--scaling for {transit_mode} ')
                    original_value = df_scaled.loc[transit_mode, transit_mode]
                    if original_value>0:
                        print(f"  Scaling mode: {transit_mode}")
                        target_original_shares = df_scaled.loc[:,transit_mode]/df_scaled.loc[transit_mode,transit_mode]
                        print(f"  original shares:\n{target_original_shares}")
                        target_value = weighted_trip_counts.loc[transit_mode, purpose]
                        print(f"  target value: {target_value}")
                        print(f"  original value: {original_value}")
                        scale_factor = target_value/original_value
                        print(f"  scale factor: {scale_factor}")
                        df_scaled.loc[transit_mode, transit_mode] *= scale_factor
                        #sum all rows excluding All row and put it there
                        df_scaled.loc['All', transit_mode] = df_scaled.loc[df_scaled.index != 'All', transit_mode].sum()
                        print('   scaling non-transit modes')
                        print(f'    non-transit modes: {df_scaled.index.difference([transit_mode, "All"])}')
                        for non_tr_mode in df_scaled.index.difference([transit_mode, 'All']):
                            print(f"     scaling {non_tr_mode}")
                            print(f"     target share: {target_original_shares[non_tr_mode]}")
                            df_scaled.loc[non_tr_mode, transit_mode] = target_original_shares[non_tr_mode]*df_scaled.loc[transit_mode, transit_mode]

                        df_scaled.loc['All', transit_mode] = df_scaled.loc[df_scaled.index != 'All', transit_mode].sum()
                    
                    else:
                        print(f"0 target")
                else:
                    print(f"purpose {purpose} dos not have {transit_mode} as tour mode")


            scaled_targets[purpose] = df_scaled
    
    return scaled_targets

scaled_trip_targets = scale_trip_targets_to_match_weighted_counts(trip_mode_target, weighted_trip_counts)

------------------------------------------------
Scaling targets for purpose: Work
--scaling for WALK-TRANSIT 
  Scaling mode: WALK-TRANSIT
  original shares:
linked_trip_mode_name
BIKE            0.008696
DRIVE_ALONE     0.034783
ESCOOTER        0.000000
KNR-TRANSIT     0.000000
PNR-TRANSIT     0.000000
SHARED2         0.078261
TAXI            0.000000
TNC-TRANSIT     0.000000
TNC_SINGLE      0.039130
WALK            0.147826
WALK-TRANSIT    1.000000
All             1.308696
Name: WALK-TRANSIT, dtype: float64
  target value: 61944.981903201
  original value: 230
  scale factor: 269.32600827478694
   scaling non-transit modes
    non-transit modes: Index(['BIKE', 'DRIVE_ALONE', 'ESCOOTER', 'KNR-TRANSIT', 'PNR-TRANSIT',
       'SHARED2', 'TAXI', 'TNC-TRANSIT', 'TNC_SINGLE', 'WALK'],
      dtype='object', name='linked_trip_mode_name')
     scaling BIKE
     target share: 0.008695652173913044
     scaling DRIVE_ALONE
     target share: 0.034782608695652174
     scaling ESCOOTER
     targe

In [22]:
# Create a new dictionary with only transit tour columns for each purpose
transit_tour_modes = ['WALK-TRANSIT', 'PNR-TRANSIT', 'KNR-TRANSIT', 'TNC-TRANSIT']
transit_only_targets = {}
for purpose, df in scaled_trip_targets.items():
    # Retain only the transit tour columns (if present)
    cols_to_keep = [col for col in transit_tour_modes if col in df.columns]
    transit_only_targets[purpose] = df[cols_to_keep].copy()

In [24]:
# Ensure all purposes have all transit trip modes as rows. Due to limited tour sample data some purposes may be missing some modes.
all_trip_modes = ['DRIVE_ALONE', 'SHARED2', 'SHARED3', 'BIKE', 'WALK', 'ESCOOTER', 'WALK-TRANSIT', 'KNR-TRANSIT', 'PNR-TRANSIT', 'TNC-TRANSIT',
       'TAXI', 'TNC_SINGLE', 'TNC_SHARED']
for purpose, df in transit_only_targets.items():
    missing_modes = [mode for mode in all_trip_modes if mode not in df.index]
    for mode in missing_modes:
        # Insert a row of zeros for the missing mode
        df.loc[mode] = 0
    # Reorder rows to match all_trip_modes + any extras (like 'All')
    row_order = [m for m in all_trip_modes if m in df.index]
    # Add an 'All' row that sums all transit trip mode rows
    all_row = df.loc[row_order].sum()
    df.loc['All'] = all_row
    row_order.append('All')
    df = df.loc[row_order]
    transit_only_targets[purpose] = df

In [25]:
# Ensure all purposes have all transit tour modes as columns, and fill missing [mode, mode] with weighted_trip_counts
for purpose, df in transit_only_targets.items():
    for mode in transit_tour_modes:
        if mode not in df.columns:
            # Create the column with zeros
            df[mode] = 0
        # If [mode, mode] is zero or missing, fill with weighted_trip_counts if available
        if (mode in df.index) and (mode in df.columns):
            if (df.loc[mode, mode] == 0 or pd.isna(df.loc[mode, mode])) and (mode in weighted_trip_counts.index) and (purpose in weighted_trip_counts.columns):
                print(f'Filling missing value for purpose {purpose}, mode {mode}')
                df.loc[mode, mode] = weighted_trip_counts.loc[mode, purpose]
    # Reorder columns to match all_trip_modes + any extras (like 'All')
    col_order = [m for m in all_trip_modes if m in df.columns]
    if 'All' in df.columns:
        col_order.append('All')
    df = df[col_order]
    # Update the 'All' row to be the sum of all other rows
    if 'All' in df.index:
        df.loc['All'] = df.loc[[m for m in all_trip_modes if m in df.index]].sum()
    transit_only_targets[purpose] = df

transit_only_targets['School']

Filling missing value for purpose Other Discretionary, mode TNC-TRANSIT
Filling missing value for purpose School, mode PNR-TRANSIT
Filling missing value for purpose School, mode KNR-TRANSIT
Filling missing value for purpose School, mode TNC-TRANSIT


tour_mode_recoded,WALK-TRANSIT,KNR-TRANSIT,PNR-TRANSIT,TNC-TRANSIT
linked_trip_mode_name,,,,
DRIVE_ALONE,0.000000,0.000000,0.000000,0.00000
SHARED2,593.842590,0.000000,0.000000,0.00000
SHARED3,0.000000,0.000000,0.000000,0.00000
BIKE,0.000000,0.000000,0.000000,0.00000
WALK,2969.212952,0.000000,0.000000,0.00000
ESCOOTER,0.000000,0.000000,0.000000,0.00000
WALK-TRANSIT,8313.796264,0.000000,0.000000,0.00000
KNR-TRANSIT,0.000000,510.632788,0.000000,0.00000
PNR-TRANSIT,0.000000,0.000000,23.782186,0.00000


## Split Individual and Joint Tour Shares

**Objective:** Separate aggregated "Other Discretionary" and "Other Maintenance" tour targets into individual (solo) and joint (group) categories for activity-based model calibration.

**Method:** The onboard survey data itself distinguishes between individual and joint travel through trip purpose classifications. The `weighted_trip_counts` DataFrame (created earlier from the OBS data) already contains separate columns for:
- `Ind-Discretionary` and `Joint-Discretionary`
- `Ind-Maintenance` and `Joint-Maintenance`

These are derived from the OBS purpose mappings where respondents indicated whether they traveled alone or with others (e.g., `Jnt_Shop`, `Jnt_Eat_Out` vs. `Ind_Shop`, `Ind_Eat_Out`).

**Implementation:** We calculate the individual vs. joint share by summing the weighted trip counts across all transit modes for each purpose type, then apply these shares to split the aggregated tour targets while preserving the total observed transit ridership.

In [43]:
# Split joint and non-joint trips using OBS data
# Calculate the individual vs. joint shares directly from the weighted_trip_counts 
# (which is based on onboard survey data, not HTS)

# Calculate discretionary split from OBS
disc_ind = weighted_trip_counts['Ind-Discretionary'].drop('All').sum()
disc_joint = weighted_trip_counts['Joint-Discretionary'].drop('All').sum()
disc_total = disc_ind + disc_joint
disc_split = disc_ind / disc_total if disc_total > 0 else 0.5

# Calculate maintenance split from OBS
maint_ind = weighted_trip_counts['Ind-Maintenance'].drop('All').sum()
maint_joint = weighted_trip_counts['Joint-Maintenance'].drop('All').sum()
maint_total = maint_ind + maint_joint
maint_split = maint_ind / maint_total if maint_total > 0 else 0.5

print(f"Discretionary split (from OBS): {disc_split:.2%} individual, {1-disc_split:.2%} joint")
print(f"Maintenance split (from OBS): {maint_split:.2%} individual, {1-maint_split:.2%} joint")

# Use disc_split and maint_split to create ind/joint maintenance/discretionary targets
# Split 'Other Discretionary' into Ind-Discretionary and Joint-Discretionary
if 'Other Discretionary' in transit_only_targets:
    disc_df = transit_only_targets['Other Discretionary']
    ind_disc = disc_df * disc_split
    joint_disc = disc_df * (1 - disc_split)
    # Recalculate 'All' row for each
    if 'All' in ind_disc.index:
        ind_disc.loc['All'] = ind_disc.loc[[i for i in ind_disc.index if i != 'All']].sum()
    if 'All' in joint_disc.index:
        joint_disc.loc['All'] = joint_disc.loc[[i for i in joint_disc.index if i != 'All']].sum()
    transit_only_targets['Ind-Discretionary'] = ind_disc
    transit_only_targets['Joint-Discretionary'] = joint_disc

# Split 'Other Maintenance' into Ind-Maintenance and Joint-Maintenance
if 'Other Maintenance' in transit_only_targets:
    maint_df = transit_only_targets['Other Maintenance']
    ind_maint = maint_df * maint_split
    joint_maint = maint_df * (1 - maint_split)
    # Recalculate 'All' row for each
    if 'All' in ind_maint.index:
        ind_maint.loc['All'] = ind_maint.loc[[i for i in ind_maint.index if i != 'All']].sum()
    if 'All' in joint_maint.index:
        joint_maint.loc['All'] = joint_maint.loc[[i for i in joint_maint.index if i != 'All']].sum()
    transit_only_targets['Ind-Maintenance'] = ind_maint
    transit_only_targets['Joint-Maintenance'] = joint_maint

transit_only_targets['Ind-Discretionary'], transit_only_targets['Joint-Discretionary'], transit_only_targets['Ind-Maintenance'], transit_only_targets['Joint-Maintenance']

Discretionary split (from OBS): 80.72% individual, 19.28% joint
Maintenance split (from OBS): 87.03% individual, 12.97% joint


(tour_mode_recoded      WALK-TRANSIT  KNR-TRANSIT  PNR-TRANSIT  TNC-TRANSIT
 linked_trip_mode_name                                                     
 DRIVE_ALONE             4333.026418     0.000000     0.000000     0.000000
 SHARED2                 1624.884907     0.000000     0.000000     0.000000
 SHARED3                    0.000000     0.000000     0.000000     0.000000
 BIKE                       0.000000     0.000000     0.000000     0.000000
 WALK                    7582.796232     0.000000     0.000000     0.000000
 ESCOOTER                   0.000000     0.000000     0.000000     0.000000
 WALK-TRANSIT           40622.122670     0.000000     0.000000     0.000000
 KNR-TRANSIT                0.000000  2076.741468     0.000000     0.000000
 PNR-TRANSIT                0.000000     0.000000  1006.291976     0.000000
 TNC-TRANSIT                0.000000     0.000000     0.000000  1038.625911
 TAXI                       0.000000     0.000000     0.000000     0.000000
 TNC_SINGLE 

In [44]:
rows = []
for purpose, df in transit_only_targets.items():
    if purpose == 'Other Maintenance' or purpose == 'Other Discretionary':
        continue
    for trip_mode in df.index:
        for tour_mode in df.columns:
            count = df.loc[trip_mode, tour_mode]
            rows.append({
                'trip_mode': trip_mode,
                'tour_mode': tour_mode,
                'counts': count,
                'purpose': purpose
            })

transit_targets_long = pd.DataFrame(rows)

# Add "All" purpose with sum of all purposes
all_purpose_rows = []
all_purpose_sum = transit_targets_long.groupby(['trip_mode', 'tour_mode'])['counts'].sum().reset_index()
all_purpose_sum['purpose'] = 'All'
transit_targets_long = pd.concat([transit_targets_long, all_purpose_sum], ignore_index=True)

# Define the desired trip_mode order
trip_mode_order = [
    'DRIVE_ALONE', 'SHARED2', 'SHARED3', 'BIKE', 'ESCOOTER', 'WALK',
    'WALK-TRANSIT', 'KNR-TRANSIT', 'PNR-TRANSIT', 'TNC-TRANSIT',
    'TNC_SINGLE', 'TNC_SHARED', 'TAXI', 'All'
]

# Create a categorical type with the specified order for sorting
transit_targets_long['trip_mode'] = pd.Categorical(
    transit_targets_long['trip_mode'], 
    categories=trip_mode_order, 
    ordered=True
)

# Sort by tour_mode and trip_mode
transit_targets_long = transit_targets_long.sort_values(['tour_mode', 'trip_mode', 'purpose']).reset_index(drop=True)

transit_targets_long.to_csv('./output/resident_trip_target.csv', index=False)

## airport targets

In [18]:
# Investigate airport_df ZIP codes to distinguish between San Diego International and CBX airports
print("Airport DataFrame shape:", airport_df.shape)
print("\nTour purposes in airport_df:")
print(airport_df['tourPurpose'].value_counts())

# Check for ZIP code columns
zip_columns = [col for col in airport_df.columns if 'ZIP' in col.upper()]
print(f"\nZIP code columns found: {zip_columns}")

# Analyze origin and destination ZIP codes
if 'ORIGIN_ADDRESS [ZIP]' in airport_df.columns:
    print("\nOrigin ZIP codes:")
    origin_zips = airport_df['ORIGIN_ADDRESS [ZIP]'].value_counts()
    print(origin_zips.head(20))
    
if 'DESTIN_ADDRESS [ZIP]' in airport_df.columns:
    print("\nDestination ZIP codes:")
    destin_zips = airport_df['DESTIN_ADDRESS [ZIP]'].value_counts()
    print(destin_zips.head(20))

# Show sample data to understand structure
print("\nSample airport_df data:")
cols_to_show = ['tourPurpose', 'ORIGIN_ADDRESS [ZIP]', 'DESTIN_ADDRESS [ZIP]']
if all(col in airport_df.columns for col in cols_to_show):
    print(airport_df[cols_to_show].head(10))

Airport DataFrame shape: (290, 208)

Tour purposes in airport_df:
Res-Airport       168
NonRes-Airport    120
Ext-Airport         2
Name: tourPurpose, dtype: int64

ZIP code columns found: ['HOME_ADDRESS [ZIP]', 'ORIGIN_ADDRESS [ZIP]', 'DESTIN_ADDRESS [ZIP]']

Origin ZIP codes:
92140    123
92101     32
92173     26
92154      8
92054      7
92109      6
92102      5
22056      3
92037      3
92056      3
92108      3
92078      3
92107      2
92055      2
91950      2
91942      2
92071      2
92115      2
91932      2
91911      2
Name: ORIGIN_ADDRESS [ZIP], dtype: int64

Destination ZIP codes:
92140    158
92173     25
92101     19
92109      5
92154      5
92115      5
92122      5
92011      3
92071      3
91942      3
22410      3
91910      3
92107      3
92121      2
91950      2
92037      2
92118      2
92054      2
92117      2
22000      2
Name: DESTIN_ADDRESS [ZIP], dtype: int64

Sample airport_df data:
         tourPurpose  ORIGIN_ADDRESS [ZIP] DESTIN_ADDRESS [ZIP]
844   

In [ ]:
# ============================================================================
# OLD METHOD - ZIP CODE BASED CLASSIFICATION (REPLACED)
# This cell is kept for reference only. 
# The new distance-based method above is now the preferred approach.
# ============================================================================

# Airport ZIP code classification
# San Diego International Airport is located in ZIP code 92101/92110 area (Liberty Station/Harbor Drive area)
# CBX (Cross Border Xpress) is located in ZIP code 92173 (Otay Mesa area near border)

# Define airport ZIP codes based on research:
# San Diego International Airport ZIP codes: 92101, 92110, and nearby airport area codes
san_diego_airport_zips = ['92101', '92110', '92106', '92107']  # Downtown/Airport area

# CBX Airport ZIP codes: 92173 is Otay Mesa where CBX is located
cbx_airport_zips = ['92173']  # Otay Mesa area

# Create airport classification function
def classify_airport_OLD_ZIP_METHOD(row):
    origin_zip = str(row['ORIGIN_ADDRESS [ZIP]']) if pd.notna(row['ORIGIN_ADDRESS [ZIP]']) else ''
    destin_zip = str(row['DESTIN_ADDRESS [ZIP]']) if pd.notna(row['DESTIN_ADDRESS [ZIP]']) else ''
    
    # Check if either origin or destination is at an airport
    if origin_zip in san_diego_airport_zips or destin_zip in san_diego_airport_zips:
        return 'San Diego International'
    elif origin_zip in cbx_airport_zips or destin_zip in cbx_airport_zips:
        return 'CBX (Cross Border)'
    else:
        # For ZIP code 92140 (which appears frequently), need to determine context
        # 92140 is not directly at either airport, so we'll classify based on other factors
        # Most 92140 trips are likely to San Diego International (the main airport)
        if origin_zip == '92140' or destin_zip == '92140':
            return 'San Diego International'  # Default assumption for now
        else:
            return 'Unknown Airport'

# NOTE: This old method is no longer used - see distance-based method above

Airport classification results:
San Diego International    237
CBX (Cross Border)          51
Unknown Airport              2
Name: Airport_Type, dtype: int64

Breakdown by tour purpose and airport type:
Airport_Type    CBX (Cross Border)  San Diego International  Unknown Airport  \
tourPurpose                                                                    
Ext-Airport                      0                        2                0   
NonRes-Airport                  45                       75                0   
Res-Airport                      6                      160                2   
All                             51                      237                2   

Airport_Type    All  
tourPurpose          
Ext-Airport       2  
NonRes-Airport  120  
Res-Airport     168  
All             290  


C:\Users\ali.etezady\AppData\Local\Temp\ipykernel_48736\2497005266.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  airport_df['Airport_Type'] = airport_df.apply(classify_airport, axis=1)


In [20]:
# Further investigate ZIP code 92140 and refine classification
print("Investigating ZIP code 92140 (most frequent):")
zip_92140_trips = airport_df[
    (airport_df['ORIGIN_ADDRESS [ZIP]'] == 92140) | 
    (airport_df['DESTIN_ADDRESS [ZIP]'] == 92140)
]
print(f"Number of trips involving 92140: {len(zip_92140_trips)}")

# Check what other ZIP codes are paired with 92140
print("\nOther ZIP codes when origin is 92140:")
other_origins = zip_92140_trips[zip_92140_trips['ORIGIN_ADDRESS [ZIP]'] == 92140]['DESTIN_ADDRESS [ZIP]'].value_counts()
print(other_origins.head(10))

print("\nOther ZIP codes when destination is 92140:")
other_destins = zip_92140_trips[zip_92140_trips['DESTIN_ADDRESS [ZIP]'] == 92140]['ORIGIN_ADDRESS [ZIP]'].value_counts()
print(other_destins.head(10))

# Based on analysis, 92140 appears to be the main airport terminal area
# Let's refine our airport classification

def refined_classify_airport(row):
    origin_zip = str(row['ORIGIN_ADDRESS [ZIP]']) if pd.notna(row['ORIGIN_ADDRESS [ZIP]']) else ''
    destin_zip = str(row['DESTIN_ADDRESS [ZIP]']) if pd.notna(row['DESTIN_ADDRESS [ZIP]']) else ''
    
    # CBX Airport (Cross Border Xpress) - ZIP 92173 is Otay Mesa
    if origin_zip == '92173' or destin_zip == '92173':
        return 'CBX'
    # San Diego International Airport - includes main terminal area and nearby
    elif (origin_zip in ['92140', '92101', '92110', '92106', '92107'] or 
          destin_zip in ['92140', '92101', '92110', '92106', '92107']):
        return 'San Diego International'
    else:
        return 'Unknown'

# Apply refined classification
airport_df_copy = airport_df.copy()
airport_df_copy['Airport_Type'] = airport_df_copy.apply(refined_classify_airport, axis=1)

print("\nRefined airport classification:")
print(airport_df_copy['Airport_Type'].value_counts())

# Create separate DataFrames for each airport
san_diego_airport_df = airport_df_copy[airport_df_copy['Airport_Type'] == 'San Diego International'].copy()
cbx_airport_df = airport_df_copy[airport_df_copy['Airport_Type'] == 'CBX'].copy()

print(f"\nSan Diego International Airport trips: {len(san_diego_airport_df)}")
print(f"CBX Airport trips: {len(cbx_airport_df)}")

# Show breakdown by purpose for each airport
print("\nSan Diego International Airport by purpose:")
print(san_diego_airport_df['tourPurpose'].value_counts())

print("\nCBX Airport by purpose:")
print(cbx_airport_df['tourPurpose'].value_counts())

Investigating ZIP code 92140 (most frequent):
Number of trips involving 92140: 123

Other ZIP codes when origin is 92140:
92173    25
92101    16
92115     5
92122     5
92109     5
92154     5
92071     3
91942     3
92011     3
91910     3
Name: DESTIN_ADDRESS [ZIP], dtype: int64

Other ZIP codes when destination is 92140:
Series([], Name: ORIGIN_ADDRESS [ZIP], dtype: int64)

Refined airport classification:
San Diego International    237
CBX                         51
Unknown                      2
Name: Airport_Type, dtype: int64

San Diego International Airport trips: 237
CBX Airport trips: 51

San Diego International Airport by purpose:
Res-Airport       160
NonRes-Airport     75
Ext-Airport         2
Name: tourPurpose, dtype: int64

CBX Airport by purpose:
NonRes-Airport    45
Res-Airport        6
Name: tourPurpose, dtype: int64


In [21]:
# Final airport analysis summary and data export
print("="*60)
print("AIRPORT CLASSIFICATION SUMMARY")
print("="*60)

print(f"\nTotal airport trips: {len(airport_df_copy)}")
print(f"San Diego International Airport: {len(san_diego_airport_df)} trips ({len(san_diego_airport_df)/len(airport_df_copy)*100:.1f}%)")
print(f"CBX (Cross Border Xpress): {len(cbx_airport_df)} trips ({len(cbx_airport_df)/len(airport_df_copy)*100:.1f}%)")

print("\nKey findings:")
print("• ZIP code 92140 appears to be the main San Diego International Airport terminal area")
print("• ZIP code 92173 is associated with CBX airport in Otay Mesa")
print("• Most NonRes-Airport trips to CBX are likely cross-border travelers")
print("• Most Res-Airport trips are to San Diego International")

# Create mode and weight analysis for each airport
print("\n" + "="*40)
print("TRANSPORTATION MODE ANALYSIS")
print("="*40)

if 'mode' in san_diego_airport_df.columns:
    print("\nSan Diego International Airport - Mode Distribution:")
    sandiego_airport_targets = san_diego_airport_df['mode'].value_counts()
    print(sandiego_airport_targets)
if 'mode' in cbx_airport_df.columns:
    print("\nCBX Airport - Mode Distribution:")
    cbx_airport_targets = cbx_airport_df['mode'].value_counts()
    print(cbx_airport_targets)

# Weight analysis
if 'LINKED_WGHT_FCTR' in san_diego_airport_df.columns:
    print("\n" + "="*40)
    print("WEIGHTED TRIP ANALYSIS")
    print("="*40)
    
    sd_weighted = san_diego_airport_df['LINKED_WGHT_FCTR'].sum()
    cbx_weighted = cbx_airport_df['LINKED_WGHT_FCTR'].sum()
    
    print(f"\nWeighted trips:")
    print(f"San Diego International: {sd_weighted:,.0f}")
    print(f"CBX: {cbx_weighted:,.0f}")
    print(f"Total: {sd_weighted + cbx_weighted:,.0f}")

# Store the classified data for further analysis
airport_classifications = {
    'all_airports': airport_df_copy,
    'san_diego_international': san_diego_airport_df,
    'cbx': cbx_airport_df
}

print(f"\nAirport data successfully classified and stored in 'airport_classifications' dictionary")

AIRPORT CLASSIFICATION SUMMARY

Total airport trips: 290
San Diego International Airport: 237 trips (81.7%)
CBX (Cross Border Xpress): 51 trips (17.6%)

Key findings:
• ZIP code 92140 appears to be the main San Diego International Airport terminal area
• ZIP code 92173 is associated with CBX airport in Otay Mesa
• Most NonRes-Airport trips to CBX are likely cross-border travelers
• Most Res-Airport trips are to San Diego International

TRANSPORTATION MODE ANALYSIS

San Diego International Airport - Mode Distribution:
WALK-TRANSIT    163
KNR-TRANSIT      42
TNC-TRANSIT      23
PNR-TRANSIT       9
Name: mode, dtype: int64

CBX Airport - Mode Distribution:
WALK-TRANSIT    39
TNC-TRANSIT      9
KNR-TRANSIT      3
Name: mode, dtype: int64

WEIGHTED TRIP ANALYSIS

Weighted trips:
San Diego International: 1,142
CBX: 290
Total: 1,431

Airport data successfully classified and stored in 'airport_classifications' dictionary


In [ ]:
# For San Diego International
sand_airport_targets = pd.crosstab(
    san_diego_airport_df['tourPurpose'],
    san_diego_airport_df['mode'],
    values=san_diego_airport_df['LINKED_WGHT_FCTR'] if 'LINKED_WGHT_FCTR' in san_diego_airport_df.columns else None,
    aggfunc='sum',
    dropna=False,
    margins=True
).fillna(0)

# For CBX
cbx_airport_targets = pd.crosstab(
    cbx_airport_df['tourPurpose'],
    cbx_airport_df['mode'],
    values=cbx_airport_df['LINKED_WGHT_FCTR'] if 'LINKED_WGHT_FCTR' in cbx_airport_df.columns else None,
    aggfunc='sum',
    dropna=False,
    margins=True
).fillna(0)

# Show the targets
print('San Diego International Airport targets:')
print(sand_airport_targets)
print('\nCBX Airport targets:')
print(cbx_airport_targets)


San Diego International Airport targets:
mode            KNR-TRANSIT  PNR-TRANSIT  TNC-TRANSIT  WALK-TRANSIT  \
tourPurpose                                                           
Ext-Airport        0.000000     5.106044     0.000000      1.334329   
NonRes-Airport    88.206036     9.389711    50.734503    225.090881   
Res-Airport      157.669564    38.618873    89.730693    476.080755   
All              245.875600    53.114629   140.465196    702.505965   

mode                    All  
tourPurpose                  
Ext-Airport        6.440372  
NonRes-Airport   373.421132  
Res-Airport      762.099885  
All             1141.961389  

CBX Airport targets:
mode            KNR-TRANSIT  TNC-TRANSIT  WALK-TRANSIT         All
tourPurpose                                                       
NonRes-Airport    16.032273     72.91283    168.604375  257.549478
Res-Airport        4.507795      0.00000     27.474619   31.982414
All               20.540068     72.91283    196.078994  289.53

In [ ]:
sand_airport_targets.to_csv('./output/sanairport_target.csv')
cbx_airport_targets.to_csv('./output/cbxairport_target.csv')


## Visitor

In [138]:
# Visitor tour mode targets by tourPurpose using tourweight
visitor_tour_targets = pd.crosstab(visitor_df['tourPurpose'], visitor_df['mode'], values=visitor_df['tourweight'], aggfunc='sum', dropna=False).fillna(0)
print('Visitor Tour Mode Targets (weighted by tourweight):')
display(visitor_tour_targets)

# Visitor trip mode targets by tourPurpose using LINKED_WGHT_FCTR
visitor_trip_targets = pd.crosstab(visitor_df['tourPurpose'], visitor_df['mode'], values=visitor_df['LINKED_WGHT_FCTR'], aggfunc='sum', dropna=False).fillna(0)
print('Visitor Trip Mode Targets (weighted by LINKED_WGHT_FCTR):')
display(visitor_trip_targets)

Visitor Tour Mode Targets (weighted by tourweight):


mode,KNR-TRANSIT,PNR-TRANSIT,TNC-TRANSIT,WALK-TRANSIT
tourPurpose,,,,
Visitor-Discretionary,207.139337,75.038651,195.385441,1639.782619
Visitor-EatOut,16.911118,1.646153,7.467015,179.779807
Visitor-Work,6.184987,0.000000,0.914525,46.139370


Visitor Trip Mode Targets (weighted by LINKED_WGHT_FCTR):


mode,KNR-TRANSIT,PNR-TRANSIT,TNC-TRANSIT,WALK-TRANSIT
tourPurpose,,,,
Visitor-Discretionary,414.278673,150.077302,390.770881,3656.715239
Visitor-EatOut,33.822236,3.292306,14.934030,400.908970
Visitor-Work,12.369974,0.000000,1.829049,102.890794


In [140]:
visitor_tour_targets.to_csv('./output/visitor_tour_target.csv')
visitor_trip_targets.to_csv('./output/visitor_trip_target.csv')

## xborder targets

In [78]:
xborder_df['mode'].value_counts()

WALK-TRANSIT    2958
TNC-TRANSIT      457
KNR-TRANSIT      360
PNR-TRANSIT       67
Name: mode, dtype: int64

In [84]:
xborder_targets = pd.crosstab(xborder_df['tourPurpose'],
    xborder_df['mode'],
    values=xborder_df['LINKED_WGHT_FCTR'] if 'LINKED_WGHT_FCTR' in xborder_df.columns else None,
    aggfunc='sum',
    dropna=False,
    margins=True
).fillna(0)

xborder_targets

mode,KNR-TRANSIT,PNR-TRANSIT,TNC-TRANSIT,WALK-TRANSIT,All
tourPurpose,,,,,
MR_Discretionary,910.953001,112.742265,936.784375,4419.820649,6380.300289
MR_School,178.974518,10.891309,244.632561,1048.262667,1482.761056
MR_Work,1403.955473,361.112881,1866.984315,10851.321572,14483.374241
All,2493.882992,484.746455,3048.401251,16319.404888,22346.435586


In [126]:
# Find KNR/TNC trips that do NOT originate in a border ZIP code
# List of border ZIP codes (as before)
border_zips = set(['92154', '92173', '92143'])

# Filter for KNR and TNC access modes
xborder_knr_tnc = xborder_df[xborder_df['access_mode'].isin(['KNR', 'TNC', 'PNR'])]

# Identify likely origin ZIP column (update if needed)
origin_zip_col = None
for col in xborder_knr_tnc.columns:
    if 'ORIGIN' in col.upper() and 'ZIP' in col.upper():
        origin_zip_col = col
        break
if origin_zip_col is None:
    print('No origin ZIP column found!')
else:
    # Find trips where origin ZIP is NOT a border ZIP
    not_border_origin = ~xborder_knr_tnc[origin_zip_col].astype(str).isin(border_zips)
    non_border_origin_trips = xborder_knr_tnc[not_border_origin]
    print(f'Number of KNR/TNC trips NOT originating in a border ZIP: {len(non_border_origin_trips)}')
    display(non_border_origin_trips[[origin_zip_col, 'access_mode', 'DESTIN_ADDRESS [ZIP]']])

Number of KNR/TNC trips NOT originating in a border ZIP: 137


,ORIGIN_ADDRESS [ZIP],access_mode,DESTIN_ADDRESS [ZIP]
1334,92010,KNR,92056
2010,92059,KNR,92173
2049,91950,KNR,92054
2068,22185,KNR,92084
2371,92075,KNR,92154
...,...,...,...
34244,92114,KNR,92173
34428,91977,KNR,92173
34872,92020,KNR,92173
34996,91913,KNR,92113


In [127]:
non_border_origin_trips['DESTIN_ADDRESS [ZIP]'].value_counts()

92173    93
92154    15
91910     3
92071     3
91911     3
21400     2
92101     2
92102     1
92019     1
22410     1
92020     1
92108     1
92114     1
92140     1
92056     1
92104     1
22056     1
92026     1
92025     1
91915     1
92084     1
92054     1
92113     1
Name: DESTIN_ADDRESS [ZIP], dtype: int64

In [119]:
# Find KNR/TNC trips that do NOT originate in a border ZIP code
# List of border ZIP codes (as before)
border_zips = set(['92154', '92173', '92143'])

# Filter for KNR and TNC access modes
xborder_knr_tnc = xborder_df[xborder_df['egress_mode'].isin(['KNR', 'TNC', 'PNR'])]

# Identify likely origin ZIP column (update if needed)
origin_zip_col = None
for col in xborder_knr_tnc.columns:
    if 'ORIGIN' in col.upper() and 'ZIP' in col.upper():
        origin_zip_col = col
        break
if origin_zip_col is None:
    print('No origin ZIP column found!')
else:
    # Find trips where origin ZIP is  a border ZIP
    not_border_origin = xborder_knr_tnc[origin_zip_col].astype(str).isin(border_zips)
    non_border_origin_trips = xborder_knr_tnc[not_border_origin]
    print(f'Number of KNR/TNC trips NOT originating in a border ZIP: {len(non_border_origin_trips)}')
    display(non_border_origin_trips[[origin_zip_col, 'egress_mode', 'access_mode']])

Number of KNR/TNC trips NOT originating in a border ZIP: 66


,ORIGIN_ADDRESS [ZIP],egress_mode,access_mode
7769,92154,TNC,TNC
13532,92173,TNC,KNR
16949,92154,TNC,Walk
16972,92173,TNC,Walk
16985,92154,TNC,Walk
...,...,...,...
28719,92154,TNC,Walk
29088,92154,TNC,Walk
29257,92173,TNC,Walk
29503,92173,TNC,TNC
